# pT reweighting bin determination — 07_baseline_tracker population

Two purposes on the current reference baseline (`07_baseline_tracker`, 587k jets):

1. Confirm the QCD/W/Z `fj_pt` spectrum mismatch and its effect on the trained classifier
   (same diagnostic as before, now re-run on 07 instead of 06).
2. Determine concrete log-spaced `fj_pt` bin edges for the pending reweighting fix in
   `ak8_VJets_tagger.yaml`'s `weights.reweight_vars` -- edges are chosen data-drivenly by
   merging a fine log grid until every class has enough jets per bin (not a fixed bin count
   picked in advance). `fj_msoftdrop` stays a single wide bin (unchanged decision, preserves
   the substructure discriminant).

In [ ]:
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep

hep.style.use("CMS")

PARQUET_PATH = "../../run_outputs/dataset_for_task07/ak8_hidneuron_full.parquet"

CLASS_ORDER = ["Wplus", "Wminus", "Z", "QCD"]
CLASS_COLORS = {
    "Wplus": "#5790fc",
    "Wminus": "#e42536",
    "Z": "#964a8b",
    "QCD": "#9c9ca1",
}

## 1. Load data and build class labels

Labels come from `label_Wplus`/`label_Wminus`/`label_Z`/`label_QCD` (one-hot by construction from
the extraction yaml), not from filenames.

In [ ]:
data = ak.from_parquet(PARQUET_PATH)
print(f"Total rows: {len(data)}")

masks = {
    "Wplus": ak.to_numpy(data["label_Wplus"]).astype(bool),
    "Wminus": ak.to_numpy(data["label_Wminus"]).astype(bool),
    "Z": ak.to_numpy(data["label_Z"]).astype(bool),
    "QCD": ak.to_numpy(data["label_QCD"]).astype(bool),
}
label_sum = sum(m.astype(int) for m in masks.values())
assert (label_sum == 1).all(), "labels not one-hot -- investigate before trusting these plots"
print("Class yields:", {c: int(m.sum()) for c, m in masks.items()})

## 2. fj_pt / fj_eta / fj_msoftdrop per class

fj_pt on a log y-axis to see the tail behavior -- this is the plot that matters for the
spectrum-matching question.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 6))
pt_bins = np.linspace(200, 3000, 60)
eta_bins = np.linspace(-2.5, 2.5, 40)
msd_bins = np.linspace(40, 650, 60)
for c in CLASS_ORDER:
    m = masks[c]
    axes[0].hist(ak.to_numpy(data["fj_pt"][m]), bins=pt_bins, histtype="step", density=True, label=c, color=CLASS_COLORS[c], linewidth=2)
    axes[1].hist(ak.to_numpy(data["fj_eta"][m]), bins=eta_bins, histtype="step", density=True, label=c, color=CLASS_COLORS[c], linewidth=2)
    axes[2].hist(ak.to_numpy(data["fj_msoftdrop"][m]), bins=msd_bins, histtype="step", density=True, label=c, color=CLASS_COLORS[c], linewidth=2)
axes[0].set_xlabel("fj_pt [GeV]"); axes[0].set_yscale("log"); axes[0].legend()
axes[1].set_xlabel("fj_eta"); axes[1].legend()
axes[2].set_xlabel("fj_msoftdrop [GeV]"); axes[2].legend()
fig.tight_layout()

## 3. Per-class recall / QCD-confusion vs. fj_pt bin

Uses `07_baseline_tracker/mlp_test_scores.parquet` to check whether the trained classifier actually
exploits the pT mismatch (rising QCD-confusion for true W/Z jets at high pT), rather than just
assuming it from the input spectrum shape alone.

In [ ]:
scores_data = ak.from_parquet("../../run_outputs/07_baseline_tracker/mlp_test_scores.parquet")
scores = ak.to_numpy(scores_data["scores"])
labels = ak.to_numpy(scores_data["_label_"])
fj_pt = ak.to_numpy(scores_data["fj_pt"])
preds = scores.argmax(axis=1)
QCD_IDX = CLASS_ORDER.index("QCD")

pt_edges = [200, 500, 700, 1000, 3100]
print(f"Test set: {len(labels)} jets\n")
print(f'{"pt bin":>15} {"class":>8} {"n":>6} {"recall":>8} {"pred_as_QCD":>12} {"mean_QCDscore":>14}')
for lo, hi in zip(pt_edges[:-1], pt_edges[1:]):
    in_bin = (fj_pt >= lo) & (fj_pt < hi)
    for ci, cname in enumerate(CLASS_ORDER):
        m = in_bin & (labels == ci)
        n = m.sum()
        if n == 0:
            continue
        recall = (preds[m] == ci).mean()
        pred_qcd = (preds[m] == QCD_IDX).mean()
        mean_qcd_score = scores[m, QCD_IDX].mean()
        print(f"{lo:>6}-{hi:<6} {cname:>8} {n:>6} {recall:>8.3f} {pred_qcd:>12.3f} {mean_qcd_score:>14.3f}")
    print()

## 4. Data-driven fj_pt bin edges for reweighting

Method: build a fine log-spaced grid over `[200, 2500]` GeV (the existing `reweight_bins` range),
then greedily merge adjacent fine bins left-to-right until every class's accumulated count in the
merged bin reaches a statistics floor. This keeps bins narrow where the spectrum is abundant (the
peak, ~400-700 GeV) and lets them widen automatically in the sparse high-pT tail -- log spacing plus
a statistics floor, rather than a fixed bin count chosen in advance. `fj_msoftdrop` is left as a
single wide bin (unchanged from `06`/`07`).

Only 22/587k jets (0.004%) have `fj_pt > 2500`, so the existing upper edge is fine as-is.

In [ ]:
def greedy_merge(fine_edges, class_counts, floor):
    """Merge adjacent fine-grid bins left-to-right until every class's running count >= floor."""
    merged = [fine_edges[0]]
    acc = {c: 0 for c in class_counts}
    for i in range(len(fine_edges) - 1):
        for c in class_counts:
            acc[c] += class_counts[c][i]
        if min(acc.values()) >= floor:
            merged.append(fine_edges[i + 1])
            acc = {c: 0 for c in class_counts}
    if merged[-1] != fine_edges[-1]:
        merged[-1] = fine_edges[-1]  # leftover tail jets get folded into the last bin
    return merged


PT_LO, PT_HI = 200, 2500
fine_edges = np.geomspace(PT_LO, PT_HI, 101)  # 100 fine log bins; the merge coarsens as needed
class_pt = {c: ak.to_numpy(data["fj_pt"][masks[c]]) for c in CLASS_ORDER}
fine_counts = {c: np.histogram(class_pt[c], bins=fine_edges)[0] for c in CLASS_ORDER}

print("Bin-count vs. statistics-floor tradeoff (fewer, wider bins as the floor rises):\n")
for floor in [1000, 2000, 3000, 5000, 8000, 10000]:
    edges = greedy_merge(fine_edges, fine_counts, floor)
    print(f"  floor={floor:>6}: {len(edges)-1:>2} bins")

In [ ]:
FLOOR = 5000  # lands at ~13 bins, matching the ballpark discussed; stable under fine-grid resolution

edges = [round(e) for e in greedy_merge(fine_edges, fine_counts, FLOOR)]
final_counts = {c: np.histogram(class_pt[c], bins=edges)[0] for c in CLASS_ORDER}

print(f"floor={FLOOR} -> {len(edges)-1} bins")
print("edges:", edges)
print()
print(f'{"lo":>6} {"hi":>6} ' + " ".join(f"{c:>8}" for c in CLASS_ORDER) + f' {"min":>8}')
for i in range(len(edges) - 1):
    row = [int(final_counts[c][i]) for c in CLASS_ORDER]
    print(f"{edges[i]:>6} {edges[i+1]:>6} " + " ".join(f"{v:>8}" for v in row) + f" {min(row):>8}")

## Next step

These edges are a candidate for `ak8_VJets_tagger.yaml`'s `weights.reweight_vars: fj_pt`
(replacing the current 2-edge/1-bin no-op), with `fj_msoftdrop` left as-is. Not yet written to
the yaml -- pending confirmation before editing the config.